# Demo Gradio: VGG16 + Random Forest phân loại loài chim

Notebook này dùng cho **Google Colab**. Luồng xử lý:

1. Mount Google Drive
2. Giải nén dataset `.zip` từ Drive về `/content`
3. Tạo `class_names.json` từ thư mục `train`
4. Load model Random Forest `.joblib` từ Drive
5. Khởi tạo VGG16 feature extractor giống notebook train: `include_top=False + GlobalAveragePooling2D`
6. Chạy demo Gradio

Bạn cần chuẩn bị trên Drive:

```text
MyDrive/
├── bird_dataset.zip
└── bird_demo/
    └── vgg16_rf.joblib
```

Nếu tên file hoặc thư mục khác, sửa lại ở **Cell 3**.


## Cell 1: Cài Gradio

In [1]:
!pip install gradio -q

## Cell 2: Mount Google Drive

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Cell 3: Khai báo đường dẫn

Sửa `ZIP_PATH`, `DEMO_DIR`, `RF_MODEL_PATH` nếu file của bạn nằm ở vị trí khác.

In [3]:
import os

# File dataset zip trên Google Drive
ZIP_PATH = "/content/drive/MyDrive/CS231/Dataset/augmented_dataset.zip"

# Thư mục giải nén dataset trong Colab
EXTRACT_DIR = "/content/bird_dataset"

# Thư mục chứa model demo trên Drive
DEMO_DIR = "/content/drive/MyDrive/CS231/DEMO"

# File Random Forest đã train
RF_MODEL_PATH = os.path.join(DEMO_DIR, "vgg16_rf_augmented.joblib")

# File class names sẽ được tạo/lưu ở đây
CLASS_NAMES_PATH = os.path.join(DEMO_DIR, "class_names.json")

print("ZIP_PATH:", ZIP_PATH)
print("EXTRACT_DIR:", EXTRACT_DIR)
print("RF_MODEL_PATH:", RF_MODEL_PATH)
print("CLASS_NAMES_PATH:", CLASS_NAMES_PATH)

print("\nKiểm tra file:")
print("Dataset zip exists:", os.path.exists(ZIP_PATH))
print("RF model exists:", os.path.exists(RF_MODEL_PATH))

ZIP_PATH: /content/drive/MyDrive/CS231/Dataset/augmented_dataset.zip
EXTRACT_DIR: /content/bird_dataset
RF_MODEL_PATH: /content/drive/MyDrive/CS231/DEMO/vgg16_rf_augmented.joblib
CLASS_NAMES_PATH: /content/drive/MyDrive/CS231/DEMO/class_names.json

Kiểm tra file:
Dataset zip exists: True
RF model exists: True


## Cell 4: Giải nén dataset từ Drive về `/content`

In [4]:
import zipfile
import os
import shutil

# Nếu muốn giải nén lại từ đầu, bật dòng dưới:
# shutil.rmtree(EXTRACT_DIR, ignore_errors=True)

os.makedirs(EXTRACT_DIR, exist_ok=True)

with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
    zip_ref.extractall(EXTRACT_DIR)

print("Đã giải nén xong dataset vào:", EXTRACT_DIR)
print("\nCác thư mục bên trong:")
print(os.listdir(EXTRACT_DIR)[:20])

Đã giải nén xong dataset vào: /content/bird_dataset

Các thư mục bên trong:
['val', 'train', 'test']


## Cell 5: Tự tìm đúng thư mục `train`

Cell này xử lý cả trường hợp zip bị lồng thêm một cấp thư mục.

In [5]:
import os

def find_train_dir(root_dir):
    for root, dirs, files in os.walk(root_dir):
        if "train" in dirs:
            return os.path.join(root, "train")
    return None

TRAIN_DIR = find_train_dir(EXTRACT_DIR)

if TRAIN_DIR is None:
    raise FileNotFoundError("Không tìm thấy thư mục train trong dataset đã giải nén. Hãy kiểm tra lại file zip.")

DATASET_ROOT = os.path.dirname(TRAIN_DIR)
VAL_DIR = os.path.join(DATASET_ROOT, "val")
TEST_DIR = os.path.join(DATASET_ROOT, "test")

print("DATASET_ROOT:", DATASET_ROOT)
print("TRAIN_DIR:", TRAIN_DIR)
print("VAL_DIR:", VAL_DIR)
print("TEST_DIR:", TEST_DIR)

print("\nKiểm tra:")
print("train exists:", os.path.exists(TRAIN_DIR))
print("val exists:", os.path.exists(VAL_DIR))
print("test exists:", os.path.exists(TEST_DIR))

DATASET_ROOT: /content/bird_dataset
TRAIN_DIR: /content/bird_dataset/train
VAL_DIR: /content/bird_dataset/val
TEST_DIR: /content/bird_dataset/test

Kiểm tra:
train exists: True
val exists: True
test exists: True


## Cell 6: Tạo `class_names.json` từ thư mục `train`

Thứ tự class dùng `sorted(...)`, giống cách TensorFlow thường lấy class theo alphabet khi dùng `image_dataset_from_directory`.

In [ ]:
import json
import os

class_names = sorted([
    name for name in os.listdir(TRAIN_DIR)
    if os.path.isdir(os.path.join(TRAIN_DIR, name))
])

print("Số lớp:", len(class_names))
print("Một vài lớp đầu:", class_names[:10])

os.makedirs(DEMO_DIR, exist_ok=True)

with open(CLASS_NAMES_PATH, "w", encoding="utf-8") as f:
    json.dump(class_names, f, ensure_ascii=False, indent=2)

print("Đã lưu class_names.json tại:", CLASS_NAMES_PATH)

Số lớp: 150
Một vài lớp đầu: ['ABBOTTS BOOBY', 'ABYSSINIAN GROUND HORNBILL', 'AFRICAN PIED HORNBILL', 'AFRICAN PYGMY GOOSE', 'ALPINE CHOUGH', 'AMERICAN AVOCET', 'AMERICAN BITTERN', 'AMERICAN DIPPER', 'AMERICAN PIPIT', 'AMERICAN WIGEON']
Đã lưu class_names.json tại: /content/drive/MyDrive/CS231/DEMO/class_names.json


## Cell 7: Import thư viện demo

In [6]:
import json
import time
import joblib
import numpy as np
import gradio as gr
import tensorflow as tf

from PIL import Image
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import VGG16
from tensorflow.keras.applications.vgg16 import preprocess_input

## Cell 8: Kiểm tra GPU

Nếu chưa có GPU: `Runtime → Change runtime type → Hardware accelerator → GPU`.

In [ ]:
print("TensorFlow version:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

TensorFlow version: 2.19.0
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


## Cell 9: Load Random Forest và class names

In [7]:
rf_model = joblib.load(RF_MODEL_PATH)

with open(CLASS_NAMES_PATH, "r", encoding="utf-8") as f:
    class_names = json.load(f)

print("Load Random Forest thành công!")
print("Số lớp:", len(class_names))
print("Một vài lớp đầu:", class_names[:10])

Load Random Forest thành công!
Số lớp: 150
Một vài lớp đầu: ['ABBOTTS BOOBY', 'ABYSSINIAN GROUND HORNBILL', 'AFRICAN PIED HORNBILL', 'AFRICAN PYGMY GOOSE', 'ALPINE CHOUGH', 'AMERICAN AVOCET', 'AMERICAN BITTERN', 'AMERICAN DIPPER', 'AMERICAN PIPIT', 'AMERICAN WIGEON']


## Cell 10: Khởi tạo VGG16 Feature Extractor

Cell này giống hướng extract trong notebook train:

```text
VGG16 include_top=False + GlobalAveragePooling2D → vector 512 chiều
```

In [8]:
base_model = VGG16(
    include_top=False,
    weights="imagenet",
    input_shape=(224, 224, 3)
)

base_model.trainable = False

inputs = layers.Input(shape=(224, 224, 3))
x = base_model(inputs, training=False)
features = layers.GlobalAveragePooling2D()(x)

feature_extractor = Model(inputs=inputs, outputs=features)

print("Đã khởi tạo VGG16 feature extractor!")
print("Output shape:", feature_extractor.output_shape)

58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Đã khởi tạo VGG16 feature extractor!
Output shape: (None, 512)


## Cell 11: Hàm dự đoán ảnh

In [9]:
def predict_bird(img):
    if img is None:
        return {}, "Vui lòng upload ảnh chim."

    start_time = time.perf_counter()

    # 1. Đảm bảo ảnh RGB
    img = img.convert("RGB")

    # 2. Resize đúng như lúc train
    img = img.resize((224, 224))

    # 3. Chuyển ảnh sang numpy array
    img_array = np.array(img).astype(np.float32)

    # 4. Thêm batch dimension: (224,224,3) -> (1,224,224,3)
    img_array = np.expand_dims(img_array, axis=0)

    # 5. Tiền xử lý theo chuẩn VGG16
    img_array = preprocess_input(img_array)

    # 6. Trích xuất đặc trưng bằng VGG16
    feature_vector = feature_extractor.predict(img_array, verbose=0)

    # 7. Random Forest dự đoán
    pred_id = rf_model.predict(feature_vector)[0]
    pred_name = class_names[int(pred_id)]

    # 8. Lấy độ tin cậy top 5
    if hasattr(rf_model, "predict_proba"):
        proba = rf_model.predict_proba(feature_vector)[0]

        top_indices = np.argsort(proba)[::-1][:5]

        result = {
            class_names[int(i)]: float(proba[i])
            for i in top_indices
        }

        confidence = float(np.max(proba))
    else:
        result = {pred_name: 1.0}
        confidence = 1.0

    end_time = time.perf_counter()
    inference_time_ms = (end_time - start_time) * 1000

    info = (
        f"Loài dự đoán: {pred_name}\n"
        f"Độ tin cậy: {confidence * 100:.2f}%\n"
        f"Thời gian xử lý: {inference_time_ms:.2f} ms\n"
        f"Feature vector shape: {feature_vector.shape}"
    )

    return result, info

## Cell 12: Test nhanh bằng ảnh upload

Nên chạy cell này trước để kiểm tra model hoạt động trước khi mở Gradio.

In [ ]:
from google.colab import files

uploaded = files.upload()

img_path = list(uploaded.keys())[0]
img = Image.open(img_path)

result, info = predict_bird(img)

print(info)
print(result)

Saving AnhProcess.png to AnhProcess.png
Loài dự đoán: DOUBLE EYED FIG PARROT
Độ tin cậy: 4.39%
Thời gian xử lý: 3216.17 ms
Feature vector shape: (1, 512)
{'DOUBLE EYED FIG PARROT': 0.043858585102097655, 'BLUE DACNIS': 0.042510695876745365, 'MILITARY MACAW': 0.03933307629305874, 'PARADISE TANAGER': 0.02969473885449062, 'GREEN MAGPIE': 0.024513638452851847}


## Cell 13: Tạo giao diện Gradio

In [14]:
def search_bird_classes(keyword):
    """
    Tìm kiếm tên loài chim trong danh sách class_names.
    Nếu chưa nhập gì thì hiển thị toàn bộ danh sách loài chim.
    """
    if keyword is None or keyword.strip() == "":
        all_classes = "\n".join(class_names)
        return f"Danh sách {len(class_names)} loài chim mô hình hỗ trợ:\n\n{all_classes}"

    keyword = keyword.strip().lower()

    matches = [
        name for name in class_names
        if keyword in name.lower()
    ]

    if len(matches) == 0:
        return (
            f"Không tìm thấy loài chim chứa từ khóa: '{keyword}'.\n"
            "Có thể loài chim này không nằm trong 150 lớp mà mô hình hỗ trợ."
        )

    result = "\n".join(matches)

    return f"Tìm thấy {len(matches)} kết quả:\n\n{result}"


with gr.Blocks(title="Hệ Thống Phân Loại Chi Tiết 150 Loài Chim") as demo:
    gr.Markdown(
        """
        # Hệ Thống Phân Loại Chi Tiết 150 Loài Chim 🐦

        Đồ án môn học CS231: Ứng dụng trích xuất đặc trưng bằng **VGG16**
        và phân loại bằng **Random Forest**.

        **Lưu ý:** Mô hình chỉ dự đoán trong danh sách 150 loài chim đã được huấn luyện.
        """
    )

    with gr.Row():
        with gr.Column(scale=1):
            input_img = gr.Image(
                label="Tải ảnh chim cần phân loại",
                type="pil"
            )

            btn = gr.Button("Dự đoán", variant="primary")

        with gr.Column(scale=1):
            output_label = gr.Label(
                label="Kết quả dự đoán Top 5",
                num_top_classes=5
            )

            output_text = gr.Textbox(
                label="Thông tin chi tiết",
                lines=5
            )

    gr.Markdown("## Tra cứu danh sách loài chim mô hình hỗ trợ")

    with gr.Row():
        search_box = gr.Textbox(
            label="Nhập tên loài chim cần tìm",
            placeholder="Ví dụ: owl, sparrow, eagle, robin..."
        )

    search_result = gr.Textbox(
    label="Kết quả tìm kiếm",
    lines=18,
    value=search_bird_classes("")
    )

    btn.click(
        fn=predict_bird,
        inputs=input_img,
        outputs=[output_label, output_text]
    )

    search_box.change(
        fn=search_bird_classes,
        inputs=search_box,
        outputs=search_result
    )

    search_box.submit(
        fn=search_bird_classes,
        inputs=search_box,
        outputs=search_result
    )

## Cell 14: Chạy demo

In [15]:
demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://379ab5fbeb37b9639e.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://379ab5fbeb37b9639e.gradio.live


## Ghi chú

Nếu file zip của bạn không tên `bird_dataset.zip`, sửa trong Cell 3:

```python
ZIP_PATH = "/content/drive/MyDrive/ten_file_dataset_cua_ban.zip"
```

Nếu model của bạn không tên `vgg16_rf.joblib`, sửa trong Cell 3:

```python
RF_MODEL_PATH = os.path.join(DEMO_DIR, "ten_model_cua_ban.joblib")
```

Nếu lỗi dự đoán sai tên lớp, hãy kiểm tra thứ tự `class_names` có giống lúc train không. Với `image_dataset_from_directory`, thứ tự thường là alphabet, nên `sorted(...)` là phù hợp.
